# TCGA Gene Expression + Reactome Graph Preparation (for BINN-as-GNN)

## Main goal

Prepare **cached, reusable artifacts** for graph-based learning on TCGA gene expression using Reactome pathways.

This notebook does **data + graph preparation only**. It intentionally stops before model training.

---

## What we download

### TCGA (UCSC Xena Toil hub)
- **Expression**: `tcga_RSEM_gene_tpm.gz` (genes as rows, samples as columns)
- **Phenotype**: `TcgaTargetGTEX_phenotype.txt.gz` (sample metadata, includes `_sample_type`)

### Reactome
- **Gene-to-pathway mapping**: `Ensembl2Reactome_All_Levels.txt`
- **Pathway hierarchy**: `ReactomePathwaysRelation.txt`

---

## What we build and save

### Dataset artifacts
- `outputs/expr_reactome_tcga_tumor_normal.parquet`  
  Expression filtered to Reactome-mapped genes and labeled TCGA samples

- `outputs/y_tcga_tumor_normal.csv`  
  Labels aligned to the expression columns

### Graph artifacts (saved under `outputs/graph/`)
- `edge_index.pt` (shape `[2, E]`, directed edges child → parent)
- `is_gene.pt` (boolean mask, length `N`)
- `node_table.csv` (node_id, node_name, node_type)
- `edge_table.csv` (src_name, dst_name, edge_type)
- `node2id.json` (mapping node_name → node_id)
- `metadata.json` (counts, filenames, parameters)

These are the only files you need to load later to start training a BINN-like GNN or a standard GNN.

---

## First task used here

We start with a simple binary classification label:

- `Primary Tumor` → 1  
- `Solid Tissue Normal` → 0

This label is highly imbalanced in TCGA. That is expected. We handle class imbalance later during training.


In [1]:
import sys
from pathlib import Path
import os

PROJECT_ROOT = Path("/home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready")

sys.path.append(str(PROJECT_ROOT / "src"))
os.environ["BINN_GNN_BASE"] = str(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
BASE_DIR = Path(os.environ.get("BINN_GNN_BASE", ".")).resolve()

Project root: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready


## 0. Environment setup

If you are running in Colab, we mount Google Drive so large files persist across sessions.


In [2]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install networkx pandas numpy scikit-learn tqdm
!pip install pytest

In [3]:
# ==== 0) Paths and environment ====
from pathlib import Path
import os, sys

# Project base directory:
# - locally: defaults to repo root (.)
# - in Colab: defaults to /content, unless you set BINN_GNN_BASE
BASE_DIR = Path(os.environ.get("BINN_GNN_BASE", ".")).resolve()

DATA_DIR = BASE_DIR / "data"
OUT_DIR  = BASE_DIR / "outputs"
GRAPH_DIR = OUT_DIR / "graph"

for d in [DATA_DIR, OUT_DIR, GRAPH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("GRAPH_DIR:", GRAPH_DIR)

# Tip (Colab): to persist outputs, set BINN_GNN_BASE to a mounted Drive folder, e.g.
#   import os
#   os.environ["BINN_GNN_BASE"] = "/content/drive/MyDrive/binn_gnn_tcga"

BASE_DIR: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready
DATA_DIR: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready/data
OUT_DIR: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready/outputs
GRAPH_DIR: /home/maani/niloo/binn_gnn_repo_ready/binn_gnn_repo_ready/outputs/graph


In [4]:
# Install lightweight dependencies (no model training yet)
!pip -q install pandas numpy tqdm networkx pyarrow

import re
import json
import gzip
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm


## 1. Download input files

We use `curl -L` (follows redirects) and skip downloads if the file already exists.


In [ ]:
URLS = {
    "tcga_expr": ("https://toil.xenahubs.net/download/tcga_RSEM_gene_tpm.gz", "tcga_RSEM_gene_tpm.gz"),
    "tcga_pheno": ("https://toil.xenahubs.net/download/TcgaTargetGTEX_phenotype.txt.gz", "TcgaTargetGTEX_phenotype.txt.gz"),
    "reactome_rel": ("https://reactome.org/download/current/ReactomePathwaysRelation.txt", "ReactomePathwaysRelation.txt"),
    "reactome_map": ("https://reactome.org/download/current/Ensembl2Reactome_All_Levels.txt", "Ensembl2Reactome_All_Levels.txt"),
}

def download(url: str, out_path: Path) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[skip] {out_path.name} already exists ({out_path.stat().st_size/1e6:.1f} MB)")
        return out_path

    print(f"[download] {out_path.name}")
    cmd = ["curl", "-L", "-o", str(out_path), url]
    subprocess.run(cmd, check=True)
    print(f"[done] {out_path.name} ({out_path.stat().st_size/1e6:.1f} MB)")
    return out_path

paths = {}
for key, (url, fname) in URLS.items():
    paths[key] = download(url, DATA_DIR / fname)

paths

[download] tcga_RSEM_gene_tpm.gz


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100    110 100    110   0      0    115      0                              0
 61 706.4M  61 434.6M   0      0  4.29M      0   02:44   01:41   01:03  2.84M

## 2. Sanity checks: file headers and formats

We check:
- expression matrix orientation (genes x samples)
- phenotype header exists
- Reactome files look like tabular text


In [ ]:
def head_gz(path: Path, n: int = 2, max_chars: int = 200):
    print(f"\n=== {path.name} (first {n} lines) ===")
    with gzip.open(path, "rt") as f:
        for _ in range(n):
            line = next(f)
            print(line[:max_chars].rstrip(), " ...")

head_gz(paths["tcga_expr"], n=2)
head_gz(paths["tcga_pheno"], n=2)

print("\nReactomePathwaysRelation.txt head:")
print(pd.read_csv(paths["reactome_rel"], sep="\t", header=None, nrows=3))

print("\nEnsembl2Reactome_All_Levels.txt head:")
print(pd.read_csv(paths["reactome_map"], sep="\t", header=None, nrows=3))



=== tcga_RSEM_gene_tpm.gz (first 2 lines) ===
sample	TCGA-19-1787-01	TCGA-S9-A7J2-01	TCGA-G3-A3CH-11	TCGA-EK-A2RE-01	TCGA-44-6778-01	TCGA-F4-6854-01	TCGA-AB-2863-03	TCGA-C8-A1HL-01	TCGA-EW-A2FS-01	TCGA-IR-A3L7-01	TCGA-05-4420-01	TCGA-R6-A8WC-01	T  ...
ENSG00000242268.2	-9.9658	0.2998	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-0.6873	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-0.8599	-0.4521  ...

=== TcgaTargetGTEX_phenotype.txt.gz (first 2 lines) ===
sample	detailed_category	primary disease or tissue	_primary_site	_sample_type	_gender	_study  ...
TCGA-V4-A9EE-01	Uveal Melanoma	Uveal Melanoma	Eye	Primary Tumor	Male	TCGA  ...

ReactomePathwaysRelation.txt head:
              0              1
0  R-BTA-109581   R-BTA-109606
1  R-BTA-109581   R-BTA-169911
2  R-BTA-109581  R-BTA-5357769

Ensembl2Reactome_All_Levels.txt head:
            0              1  \
0  2RSSE.1b.1   R-CEL-162582   
1  2RSSE.1b.1   R-CEL-194315   
2  2RSSE.1b.

## 3. Reactome mapping: ENSG genes → R-HSA pathways

The Reactome mapping file contains multiple species. We keep:
- `species == Homo sapiens`
- `pathway_id` starts with `R-HSA-`
- `ensembl_or_id` starts with `ENSG` (needed for TCGA expression)


In [ ]:
reactome_map = pd.read_csv(paths["reactome_map"], sep="\t", header=None)
reactome_map.columns = ["ensembl_or_id", "pathway_id", "url", "pathway_name", "evidence", "species"]

reactome_map["species_clean"] = reactome_map["species"].astype(str).str.strip().str.lower()
reactome_map["pathway_id"] = reactome_map["pathway_id"].astype(str)
reactome_map["ensembl_or_id"] = reactome_map["ensembl_or_id"].astype(str)

g2p = reactome_map[
    reactome_map["species_clean"].eq("homo sapiens")
    & reactome_map["pathway_id"].str.startswith("R-HSA-")
    & reactome_map["ensembl_or_id"].str.startswith("ENSG")
].copy()

g2p = g2p[["ensembl_or_id", "pathway_id"]].drop_duplicates()

print("Gene→Pathway edges:", g2p.shape[0])
print("Unique ENSG genes:", g2p["ensembl_or_id"].nunique())
print("Unique pathways:", g2p["pathway_id"].nunique())
print("Example rows:")
g2p.head()

Gene→Pathway edges: 153961
Unique ENSG genes: 13029
Unique pathways: 2799
Example rows:


,ensembl_or_id,pathway_id
851050,ENSG00000000419,R-HSA-162699
851051,ENSG00000000419,R-HSA-163125
851052,ENSG00000000419,R-HSA-1643685
851053,ENSG00000000419,R-HSA-3781865
851054,ENSG00000000419,R-HSA-392499


## 4. Reactome pathway hierarchy: parent → child (in the file)

Reactome’s `ReactomePathwaysRelation.txt` is a pathway hierarchy edge list.

- **Column 1**: parent pathway (more general)
- **Column 2**: child pathway (more specific)

For the models we want later, it is usually more convenient to use **directed edges child → parent** (bottom-up information flow).  
So we will load the file in its native order (parent, child) but build our graph edges in the **child → parent** direction.

We also keep only human pathways (`R-HSA-`).


In [ ]:
rel = pd.read_csv(paths["reactome_rel"], sep="\t", header=None, names=["parent_pathway", "child_pathway"])
rel["child_pathway"] = rel["child_pathway"].astype(str)
rel["parent_pathway"] = rel["parent_pathway"].astype(str)

rel = rel[
    rel["child_pathway"].str.startswith("R-HSA-")
    & rel["parent_pathway"].str.startswith("R-HSA-")
].drop_duplicates()

print("Human pathway relations:", rel.shape[0])
rel.head()

Human pathway relations: 2864


,parent_pathway,child_pathway
11142,R-HSA-109581,R-HSA-109606
11143,R-HSA-109581,R-HSA-169911
11144,R-HSA-109581,R-HSA-5357769
11145,R-HSA-109581,R-HSA-75153
11146,R-HSA-109582,R-HSA-140877


## 5. Load phenotype and create Tumor vs Normal labels

The phenotype file may contain non-UTF8 bytes, so we use a fallback encoding strategy.


In [ ]:
def read_tsv_gz(path: Path, sep: str = "\t", encodings=("utf-8", "latin1", "cp1252")) -> pd.DataFrame:
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, compression="gzip", encoding=enc, low_memory=False)
        except UnicodeDecodeError as e:
            last_err = e
            print(f"[warn] failed with encoding={enc}: {e}")
    raise last_err

pheno = read_tsv_gz(paths["tcga_pheno"], sep="\t")
print("Phenotype shape:", pheno.shape)
print("Columns:", pheno.columns.tolist())

pheno_tcga = pheno[pheno["_study"] == "TCGA"].copy()

keep_types = {"Primary Tumor": 1, "Solid Tissue Normal": 0}
pheno_tcga = pheno_tcga[pheno_tcga["_sample_type"].isin(keep_types)].copy()
pheno_tcga["y"] = pheno_tcga["_sample_type"].map(keep_types).astype(int)

print("TCGA tumor+normal samples:", pheno_tcga.shape[0])
print("Tumor:", (pheno_tcga["y"]==1).sum(), "Normal:", (pheno_tcga["y"]==0).sum())

pheno_tcga[["sample", "_sample_type", "y"]].head()

[warn] failed with encoding=utf-8: 'utf-8' codec can't decode byte 0xca in position 11: invalid continuation byte
Phenotype shape: (19131, 7)
Columns: ['sample', 'detailed_category', 'primary disease or tissue', '_primary_site', '_sample_type', '_gender', '_study']
TCGA tumor+normal samples: 9912
Tumor: 9185 Normal: 727


,sample,_sample_type,y
0,TCGA-V4-A9EE-01,Primary Tumor,1
1,TCGA-VD-AA8N-01,Primary Tumor,1
2,TCGA-V4-A9EI-01,Primary Tumor,1
3,TCGA-VD-AA8O-01,Primary Tumor,1
4,TCGA-WC-A888-01,Primary Tumor,1


## 6. Match phenotype samples to expression columns

The expression file is genes x samples. The header line gives us the sample columns.


In [ ]:
with gzip.open(paths["tcga_expr"], "rt") as f:
    header = f.readline().strip().split("\t")

gene_col_name = header[0]           # usually "sample" (holds gene ids)
expr_sample_cols = header[1:]
expr_samples = set(expr_sample_cols)

print("Expression columns:", len(expr_samples))
print("Gene id column name:", gene_col_name)

pheno_tcga = pheno_tcga[pheno_tcga["sample"].isin(expr_samples)].copy()

print("After matching expression columns:", pheno_tcga.shape[0])
print("Tumor:", (pheno_tcga["y"]==1).sum(), "Normal:", (pheno_tcga["y"]==0).sum())

Expression columns: 10535
Gene id column name: sample
After matching expression columns: 9912
Tumor: 9185 Normal: 727


## 7. Load expression filtered to Reactome genes (streaming)

We stream-load `tcga_RSEM_gene_tpm.gz` in chunks, keeping:
- labeled samples (from phenotype)
- genes that exist in `g2p` (Reactome-mapped ENSG ids)

Identifier harmonization:
- TCGA uses `ENSG... .<version>` (example: `ENSG00000141510.12`)
- Reactome uses versionless ENSG ids
So we strip the suffix `.12` using a simple regex.


In [ ]:
def strip_ensembl_version(x: str) -> str:
    return re.sub(r"\.\d+$", "", str(x))

reactome_gene_set = set(g2p["ensembl_or_id"].astype(str))
keep_samples = pheno_tcga["sample"].tolist()

# Optional: for faster runs while debugging, limit number of samples
MAX_SAMPLES = None  # e.g. 2000
if MAX_SAMPLES is not None and len(keep_samples) > MAX_SAMPLES:
    keep_samples = keep_samples[:MAX_SAMPLES]
    pheno_tcga = pheno_tcga[pheno_tcga["sample"].isin(keep_samples)].copy()

usecols = [gene_col_name] + keep_samples
chunksize = 3000

expr_chunks = []
for chunk in pd.read_csv(paths["tcga_expr"], sep="\t", usecols=usecols, chunksize=chunksize):
    chunk = chunk.rename(columns={gene_col_name: "gene_id_raw"})
    chunk["gene_id"] = chunk["gene_id_raw"].map(strip_ensembl_version)

    chunk = chunk[chunk["gene_id"].isin(reactome_gene_set)]
    if chunk.empty:
        continue

    chunk = chunk.drop(columns=["gene_id_raw"]).set_index("gene_id")
    expr_chunks.append(chunk)

expr = pd.concat(expr_chunks, axis=0)

# Align columns and labels
expr = expr.loc[:, pheno_tcga["sample"]]
y = pheno_tcga.set_index("sample").loc[expr.columns, "y"].astype(int)

print("Expression shape (genes x samples):", expr.shape)
print("Label counts:", y.value_counts().to_dict())
print("Tumor fraction:", float(y.mean()))

expr.iloc[:3, :5]

Expression shape (genes x samples): (11403, 9912)
Label counts: {1: 9185, 0: 727}
Tumor fraction: 0.9266545601291364


,TCGA-V4-A9EE-01,TCGA-VD-AA8N-01,TCGA-V4-A9EI-01,TCGA-VD-AA8O-01,TCGA-WC-A888-01
gene_id,,,,,
ENSG00000167578,5.4611,4.8334,5.2597,5.9081,6.3705
ENSG00000078237,3.5473,2.8178,3.5633,2.7826,2.3732
ENSG00000198242,10.5737,11.0476,10.4901,10.8085,10.5106


## 8. Save dataset artifacts

These files are the starting point for training later.


In [ ]:
expr_path = OUT_DIR / "expr_reactome_tcga_tumor_normal.parquet"
y_path = OUT_DIR / "y_tcga_tumor_normal.csv"

expr.to_parquet(expr_path)
y.to_csv(y_path)

print("Saved:")
print(" -", expr_path)
print(" -", y_path)

Saved:
 - /content/drive/MyDrive/binn_gnn_repo_ready/outputs/expr_reactome_tcga_tumor_normal.parquet
 - /content/drive/MyDrive/binn_gnn_repo_ready/outputs/y_tcga_tumor_normal.csv


## 9. Build the shared graph (genes + pathways)

Nodes:
- Gene nodes: `expr.index` (filtered ENSG ids)
- Pathway nodes: pathways connected to the kept genes, plus all ancestors in the hierarchy

Edges:
- gene → pathway (from `g2p`)
- pathway(child) → pathway(parent) (from `rel`)

We also sort edges to make the saved graph deterministic across runs.


In [ ]:
# ---- Build and save the raw graph (genes + pathways) using the package ----
from binn_gnn.graph.build_raw import build_raw_reactome_graph, save_raw_graph

# The raw graph uses genes from the expression matrix (sorted for a stable node ordering).
expr_gene_ids = expr.index.astype(str).tolist()

raw = build_raw_reactome_graph(
    expr_gene_ids=sorted(expr_gene_ids),
    g2p=g2p,
    rel=rel,
)

save_raw_graph(raw, GRAPH_DIR)

print(f"Raw graph saved to: {GRAPH_DIR}")
print("Raw graph | N:", raw.node_table.shape[0], "| E:", raw.edge_table.shape[0], "| genes:", raw.num_genes)
display(raw.edge_table["edge_type"].value_counts())

Raw graph saved to: /content/drive/MyDrive/binn_gnn_repo_ready/outputs/graph
Raw graph | N: 14201 | E: 140581 | genes: 11403


,count
edge_type,
gene_to_pathway,137767
pathway_to_parent,2814


## 10. Raw graph artifacts saved under `outputs/graph/`

The raw graph is saved in a *data-centric* format so it can be reused by different models:

- `edge_index.pt` (PyTorch tensor `[2, E]`)
- `is_gene.pt` (boolean mask `[N]`)
- `node_table.csv` (node id → name/type)
- `edge_table.csv` (src/dst + edge types)
- `node2id.json` (name → raw node id)

These files are the shared starting point for both:
- padded feedforward-style BINN layerization (copy-node padding), and
- native DAG propagation (no padding).

## 11. Done

You can now start a training notebook by loading:

- `outputs/expr_reactome_tcga_tumor_normal.parquet`
- `outputs/y_tcga_tumor_normal.csv`
- `outputs/graph/edge_index.pt`
- `outputs/graph/is_gene.pt`
- `outputs/graph/node_table.csv`

No re-downloading or re-processing is needed unless you want to change the task or filtering.


# Phase 2: Build a BINN-style layered graph (no training yet)

In Phase 1 we created a **raw biological graph**:

- **Genes** (observed features)
- **Pathways** (hidden nodes)
- Edges:
  - gene → pathway (membership from Reactome mapping)
  - pathway → parent pathway (Reactome hierarchy, used bottom-up)

That raw graph is ideal for a “standard GNN” (GCN/GAT) because GNNs can ingest an edge list directly.

However, the original BINN is a **feedforward masked network** where each layer only connects to the next layer.  
So Phase 2 creates a *layered* version of the same biology that we can later use for a **BINN-exact message passing schedule**.

## What Phase 2 produces

We will create a new folder:

- `outputs/graph_layered_binn/`

It will contain:

- `node_table_layered.csv` (layered nodes, including padded duplicates)
- `edge_table_layered.csv` (edges only between consecutive layers)
- `edge_index_layered.pt` (PyTorch tensor for later PyG use)
- `layer_info.json` (layer sizes, leaf and root nodes, etc.)

## Key idea: “raggedness” and padding

Reactome’s pathway hierarchy is a DAG with uneven depth. That creates **edges that jump multiple levels**.

To make a strict feedforward layer-to-layer network, we will **pad** these jumps by creating *duplicate copies of pathways at intermediate layers* so that every edge goes from layer ℓ → ℓ+1.


## A. Load the raw graph artifacts (from Phase 1)

We reload the raw graph tables from `outputs/graph/` and then create a *padded, strictly-layered* graph
where every edge goes from layer ℓ to ℓ+1 (the classical feedforward BINN constraint).

In [ ]:
# ---- Build and save the padded, strictly-layered BINN graph (copy-node padding) ----
import json
import pandas as pd
import torch

from binn_gnn.graph.layerize import layerize_reactome_like, save_layered_graph

RAW_GRAPH_DIR = GRAPH_DIR
LAYERED_GRAPH_DIR = OUT_DIR / "graph_layered_binn"
LAYERED_GRAPH_DIR.mkdir(parents=True, exist_ok=True)

raw_node_table = pd.read_csv(RAW_GRAPH_DIR / "node_table.csv")
raw_edge_table = pd.read_csv(RAW_GRAPH_DIR / "edge_table.csv")

layered = layerize_reactome_like(raw_node_table, raw_edge_table)
save_layered_graph(layered, LAYERED_GRAPH_DIR)

print("Layered graph saved to:", LAYERED_GRAPH_DIR)
print("L (num propagation steps):", layered.layer_info["L"])
print("Layered N:", layered.layer_info["num_layered_nodes"], "E:", layered.layer_info["num_layered_edges"])
print("Root pathways (layered ids):", len(layered.layer_info["root_pathways_layered_ids"]))

# Dataset-specific convenience tensor:
# Map expression-gene row order -> layered gene node ids (layer=0 gene nodes).
gene_rows = layered.node_table.query("node_type=='gene' and layer==0")[["orig_name","layered_id"]]
gene_to_layered = dict(zip(gene_rows["orig_name"].astype(str), gene_rows["layered_id"].astype(int)))

expr_gene_order = expr.index.astype(str).tolist()
missing = [g for g in expr_gene_order if g not in gene_to_layered]
if missing:
    raise ValueError(f"{len(missing)} expression genes missing from layered graph. Example: {missing[:5]}")

gene_layered_ids_in_expr_order = torch.tensor([gene_to_layered[g] for g in expr_gene_order], dtype=torch.long)
torch.save(gene_layered_ids_in_expr_order, LAYERED_GRAPH_DIR / "gene_layered_ids_in_expr_order.pt")

print("Saved:", LAYERED_GRAPH_DIR / "gene_layered_ids_in_expr_order.pt")

Layered graph saved to: /content/drive/MyDrive/binn_gnn_repo_ready/outputs/graph_layered_binn
L (num propagation steps): 12
Layered N: 16283 E: 44955
Root pathways (layered ids): 29
Saved: /content/drive/MyDrive/binn_gnn_repo_ready/outputs/graph_layered_binn/gene_layered_ids_in_expr_order.pt


## B. Create layered nodes (duplicate pathways to remove raggedness)

We keep genes only in **layer 0**.

Pathways have an intrinsic “base layer” from the hierarchy depth (leaf pathways start at layer 1).

To ensure a strict feedforward structure, we may need to **duplicate** a pathway across multiple layers:

- If a child pathway connects to a high-level parent, we create copies of the child pathway at intermediate layers.
- Root pathways (no parent) are padded up to the maximum layer `L` so we can read out predictions from a single layer.

The result is a new node set where each node is `(original_node_id, layer)`.


## C. Create layered edges (all edges go from layer ℓ → ℓ+1)

We create three types of edges:

1. **Gene → leaf pathway** edges (layer 0 → 1)
2. **Pathway child → pathway parent** edges (layer k → k+1), aligned to the parent’s layer
3. **Padding edges** connecting duplicates of the same pathway across layers (layer k → k+1)

After this step we verify that every edge increases the layer by exactly 1.


## D. Save the layered graph artifacts

These are the only files you will need later when you start model training.

We save:

- `node_table_layered.csv`
- `edge_table_layered.csv`
- `edge_index_layered.pt`
- `layer_info.json`


## E. Sanity checks and next step

Sanity checks you should expect:

- **Root pathways** should be a relatively small number (tens, not thousands).
- **Edge delta counts** should be `{1: ...}` only.
- **Edges per layer** should be non-zero for most layers (because padding edges fill gaps).

If any of these checks fail, fix the raw graph direction first (Phase 1, Section 4).

### Next step (separate notebook)

Now we can implement the **BINN-exact MPNN layer**:

- learn a **unique scalar weight per edge**
- perform message passing from layer 0 to layer L
- read out from `root_pathways_layered_ids` (not global pooling)

We will do that next, without starting training yet.


## F. Save sequential training artifacts for the layered BINN graph

The raw DAG is still useful for diagnostics, but the training notebook should consume the **strictly layered BINN graph** from `outputs/graph_layered_binn/`.

This cell precomputes the tensors needed by a sequential layer-by-layer MPNN:

- `node_layers.pt`
- `layer_edges.pt`
- `layer_node_ids.pt`
- `root_layered_ids.pt`

That lets the training notebook avoid dynamic masking and guarantees that message passing follows the biological hierarchy one hop at a time.

In [ ]:
# ---- Save sequential training artifacts for the layered BINN graph ----
import json
import torch
import pandas as pd

node_table_layered = pd.read_csv(LAYERED_GRAPH_DIR / "node_table_layered.csv")
edge_index_layered = torch.load(LAYERED_GRAPH_DIR / "edge_index_layered.pt").long()

with open(LAYERED_GRAPH_DIR / "layer_info.json", "r") as f:
    layer_info = json.load(f)

node_id_col = "layered_id" if "layered_id" in node_table_layered.columns else "node_id"
num_layered_nodes = int(node_table_layered[node_id_col].max()) + 1

node_layers = torch.tensor(
    node_table_layered.sort_values(node_id_col)["layer"].to_numpy(),
    dtype=torch.long
)

L = int(node_layers.max().item())
src_layers = node_layers[edge_index_layered[0]]
dst_layers = node_layers[edge_index_layered[1]]

if not torch.all(dst_layers == src_layers + 1):
    bad = torch.where(dst_layers != src_layers + 1)[0][:10].tolist()
    raise ValueError(f"Layered graph is invalid: found non-consecutive edges at indices {bad}")

layer_edges = [edge_index_layered[:, src_layers == l].clone() for l in range(L)]
layer_node_ids = [torch.where(node_layers == l)[0].clone() for l in range(L + 1)]
root_layered_ids = torch.tensor(layer_info["root_pathways_layered_ids"], dtype=torch.long)

torch.save(node_layers, LAYERED_GRAPH_DIR / "node_layers.pt")
torch.save(layer_edges, LAYERED_GRAPH_DIR / "layer_edges.pt")
torch.save(layer_node_ids, LAYERED_GRAPH_DIR / "layer_node_ids.pt")
torch.save(root_layered_ids, LAYERED_GRAPH_DIR / "root_layered_ids.pt")

print("Saved sequential artifacts:")
print(" -", LAYERED_GRAPH_DIR / "node_layers.pt")
print(" -", LAYERED_GRAPH_DIR / "layer_edges.pt")
print(" -", LAYERED_GRAPH_DIR / "layer_node_ids.pt")
print(" -", LAYERED_GRAPH_DIR / "root_layered_ids.pt")
print("Num layers:", L)
print("Layer edge counts:", [int(e.shape[1]) for e in layer_edges][:10], "...")
print("Num root readout nodes:", int(root_layered_ids.numel()))

## G. Unit tests for graph integrity

These checks are the guard rails for the training notebook.  
If any of them fails, fix the graph artifacts before touching the loss or optimizer.

In [ ]:
# ---- Unit tests: layered graph must be valid for sequential message passing ----
import networkx as nx
import torch

name_col = "orig_name" if "orig_name" in node_table_layered.columns else "node_name"

def run_graph_unit_tests():
    # 1) Still a DAG
    G = nx.DiGraph()
    G.add_nodes_from(range(num_layered_nodes))
    G.add_edges_from(edge_index_layered.t().tolist())
    assert nx.is_directed_acyclic_graph(G), "Layered graph must remain a DAG"

    # 2) Every edge advances exactly one layer
    assert torch.all(node_layers[edge_index_layered[1]] == node_layers[edge_index_layered[0]] + 1),         "Every layered edge must advance exactly one layer"

    # 3) Genes live only at layer 0
    gene_rows = node_table_layered.query("node_type == 'gene'")
    assert gene_rows["layer"].nunique() == 1 and int(gene_rows["layer"].iloc[0]) == 0,         "Gene nodes must exist only at layer 0"

    # 4) Root readout nodes are all at the final layer
    assert torch.all(node_layers[root_layered_ids] == L),         "All root readout nodes must live at the final layer"

    # 5) Saved gene ids must align exactly with expression row order
    assert len(gene_layered_ids_in_expr_order) == expr.shape[0],         "gene_layered_ids_in_expr_order length must match expression rows"

    ordered_gene_names = (
        node_table_layered
        .set_index(node_id_col)
        .loc[gene_layered_ids_in_expr_order.tolist(), name_col]
        .astype(str)
        .tolist()
    )
    assert ordered_gene_names == expr.index.astype(str).tolist(),         "Saved gene id order does not match expression row order"

    # 6) layer_edges must cover every edge exactly once
    reconstructed = torch.cat(layer_edges, dim=1)
    orig_edges = sorted(map(tuple, edge_index_layered.t().tolist()))
    recon_edges = sorted(map(tuple, reconstructed.t().tolist()))
    assert recon_edges == orig_edges, "layer_edges must partition the full layered edge list"

    # 7) Every non-input node should have at least one incoming edge
    indeg = torch.bincount(edge_index_layered[1], minlength=num_layered_nodes)
    non_input_nodes = torch.where(node_layers > 0)[0]
    assert torch.all(indeg[non_input_nodes] > 0),         "Found non-input nodes with zero indegree"

    print("All graph unit tests passed.")

run_graph_unit_tests()